In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

from sentence_transformers import SentenceTransformer

/home/ben/Bureau/Pre-production/Offres_emplois_senegal/.benv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import pandas as pd
import psycopg2

class Visualisation:
    def __init__(self, host, port, database, user, password):
        self.conn_params = {
            "host": host,
            "port": port,
            "database": database,
            "user": user,
            "password": password
        }
        self.conn = None
        self._connect()

    def _connect(self):
        try:
            self.conn = psycopg2.connect(**self.conn_params)
            print("Connexion réussie à la base de données PostgreSQL")
        except Exception as e:
            print(f" Erreur de connexion : {e}")
            self.conn = None

    def get_data(self, table_name):
        if not self.conn:
            print("Aucune connexion active.")
            return None
        
        # Sécurité basique contre l'injection SQL sur le nom de table
        if not table_name.replace("_", "").isalnum():
            print(" Nom de table invalide.")
            return None
            
        try:
            query = f'SELECT * FROM "{table_name}"'
            df = pd.read_sql_query(query, self.conn)
            print(f" {len(df)} lignes récupérées depuis '{table_name}'")
            return df
        except Exception as e:
            print(f" Erreur lors de la requête : {e}")
            return None

    def close(self):
        if self.conn:
            self.conn.close()
            print("Connexion fermée.")

In [10]:
# from connexion_warehouse import Visualisation

# 1. Initialisation avec les paramètres de connexion
viz = Visualisation(
    host="localhost",
    port=5493,
    database="datawarehouse",
    user="admin",
    password="admin_pwd"
)

# 2. Récupération des données
df = viz.get_data("offres_emploi_ml")

# 3. Vérification rapide
if df is not None:
    print("\n Aperçu des données :")
    print(df.head())
    print(f"\nDimensions : {df.shape}")

viz.close()

Connexion réussie à la base de données PostgreSQL
 35792 lignes récupérées depuis 'offres_emploi_ml'

 Aperçu des données :
         entreprise                                poste      competence  \
0  INTRA INTERIM SN  Ouvrier d’Usine - Roumanie (Europe)  Accompagnement   
1  INTRA INTERIM SN  Ouvrier d’Usine - Roumanie (Europe)  Accompagnement   
2  INTRA INTERIM SN  Ouvrier d’Usine - Roumanie (Europe)  Accompagnement   
3  INTRA INTERIM SN  Ouvrier d’Usine - Roumanie (Europe)  Accompagnement   
4  INTRA INTERIM SN  Ouvrier d’Usine - Roumanie (Europe)      Assemblage   

                         formation_clean             niveau_etude  contrat  \
0  Niveau d’anglais intermédiaire requis  Qualification avant bac      CDI   
1  Niveau d’anglais intermédiaire requis  Qualification avant bac      CDI   
2  Niveau d’anglais intermédiaire requis  Qualification avant bac  Intérim   
3  Niveau d’anglais intermédiaire requis  Qualification avant bac  Intérim   
4  Niveau d’anglais intermédi

/tmp/ipykernel_65225/4236701334.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, self.conn)


In [11]:
df["experience"].unique()

<StringArray>
[                               'Etudiant',
                   'jeune diplômé et plus',
        'Expérience entre 5 ans et 10 ans',
 'Expérience entre 2 ans et 5 ans et plus',
         'Expérience entre 2 ans et 5 ans',
                     'Expérience > 10 ans',
                           'jeune diplômé',
                'Débutant < 2 ans et plus',
                        'Débutant < 2 ans']
Length: 9, dtype: str

In [12]:
data_ml = df.drop_duplicates()
data_ml.shape

(17896, 9)

In [45]:
df.info

<bound method DataFrame.info of              entreprise                                              poste  \
0      INTRA INTERIM SN                Ouvrier d’Usine - Roumanie (Europe)   
1      INTRA INTERIM SN                Ouvrier d’Usine - Roumanie (Europe)   
2      INTRA INTERIM SN                Ouvrier d’Usine - Roumanie (Europe)   
3      INTRA INTERIM SN                Ouvrier d’Usine - Roumanie (Europe)   
4      INTRA INTERIM SN                Ouvrier d’Usine - Roumanie (Europe)   
...                 ...                                                ...   
35787    TECTRA SÉNÉGAL  Responsable ESG, Impact et Assistance Techniqu...   
35788    TECTRA SÉNÉGAL  Responsable ESG, Impact et Assistance Techniqu...   
35789    TECTRA SÉNÉGAL  Responsable ESG, Impact et Assistance Techniqu...   
35790    TECTRA SÉNÉGAL  Responsable ESG, Impact et Assistance Techniqu...   
35791    TECTRA SÉNÉGAL  Responsable ESG, Impact et Assistance Techniqu...   

               competence      

In [13]:
data_ml["experience"]

0                     Etudiant
1        jeune diplômé et plus
2                     Etudiant
3        jeune diplômé et plus
4                     Etudiant
                 ...          
17891      Expérience > 10 ans
17892      Expérience > 10 ans
17893      Expérience > 10 ans
17894      Expérience > 10 ans
17895      Expérience > 10 ans
Name: experience, Length: 17896, dtype: str

### Preparation pour le traitement Ml 

In [14]:
data_ml= df[['poste', 'competence', 'region', "date_de_publication", "experience"]]

### Fonction de clean

In [15]:
import re 
def clean_text(text):
    """
    Nettoie le texte :
    - minuscule
    - suppression caractères spéciaux
    """
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [16]:
def prepare_data(df):

    data = df[['poste', 'competence', 'region', 'experience', 'date_de_publication']].copy()

    # nettoyage
    data["competence_clean"] = data["competence"].apply(clean_text)
    data["region_clean"] = data["region"].apply(clean_text)

    # regroupement (IMPORTANT)
    grouped = data.groupby(["poste", "region_clean"])["competence_clean"] \
                  .apply(lambda x: " ".join(x)).dropna().reset_index()

    # moyenne expérience
    exp = data.groupby(["poste", "region_clean"])["experience"] \
                      .apply(lambda x: " ".join(x)).dropna().reset_index()

    grouped = grouped.merge(exp, on=["poste", "region_clean"])

    return grouped

In [6]:
df["experience"].unique()

<StringArray>
[                               'Etudiant',
                   'jeune diplômé et plus',
        'Expérience entre 5 ans et 10 ans',
 'Expérience entre 2 ans et 5 ans et plus',
         'Expérience entre 2 ans et 5 ans',
                     'Expérience > 10 ans',
                           'jeune diplômé',
                'Débutant < 2 ans et plus',
                        'Débutant < 2 ans']
Length: 9, dtype: str

In [17]:
data_prepa = prepare_data(data_ml)

data_prepa

,poste,region_clean,competence_clean,experience
0,Assistant Comptable (Stagiaire) - Dakar,dakar,comptabilit finance gestion rapprochement banc...,Débutant < 2 ans et plus Débutant < 2 ans et p...
1,Chargé de Clientèle Bilingue (Français-Anglai...,dakar,vente vente vente vente vente vente vente vent...,Etudiant jeune diplômé et plus Etudiant jeune ...
2,ANGULAR Developer (M/F),dakar,bootstrap bootstrap bootstrap css3 css3 css3 c...,Expérience entre 5 ans et 10 ans Expérience en...
3,ANGULAR Developer (M/F),diourbel,bootstrap bootstrap bootstrap css3 css3 css3 c...,Expérience entre 5 ans et 10 ans Expérience en...
4,ANGULAR Developer (M/F),fatick,bootstrap bootstrap bootstrap css3 css3 css3 c...,Expérience entre 5 ans et 10 ans Expérience en...
...,...,...,...,...
421,Z/OS System Engineer Senior Storage(H/F),fatick,cics cics cics db2 db2 db2 ibm ibm ibm jcl jcl...,Expérience entre 5 ans et 10 ans Expérience en...
422,Z/OS System Engineer Senior Storage(H/F),k dougou,cics cics cics db2 db2 db2 ibm ibm ibm jcl jcl...,Expérience entre 5 ans et 10 ans Expérience en...
423,Z/OS System Engineer Senior Storage(H/F),kaffrine,cics cics cics db2 db2 db2 ibm ibm ibm jcl jcl...,Expérience entre 5 ans et 10 ans Expérience en...
424,Z/OS System Engineer Senior Storage(H/F),kaolack,cics cics cics db2 db2 db2 ibm ibm ibm jcl jcl...,Expérience entre 5 ans et 10 ans Expérience en...


#### Modele 1

In [18]:
class TfidfJobModel:
    
    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=5000)
        self.encoder = OneHotEncoder()
        self.model = LogisticRegression(max_iter=1000)

    def fit(self, df):
        """
        Entraîne le modèle de classification
        """
        df["text"] = df["competence_clean"] + " " + df["region_clean"]

        X_text = self.vectorizer.fit_transform(df["text"])
        X_region = self.encoder.fit_transform(df[["region_clean"]])

        X = hstack([X_text, X_region])
        y = df["poste"]

        self.model.fit(X, y)

    def predict(self, skills, region):
        """
        Prédit un poste à partir des skills + région
        """
        text = clean_text(skills + " " + region)

        X_text = self.vectorizer.transform([text])
        X_region = self.encoder.transform([[clean_text(region)]])

        X = hstack([X_text, X_region])

        return self.model.predict(X)[0]

In [9]:
from zenml.steps import step
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

class TfidfJobModel:
    
    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=5000)
        self.encoder = OneHotEncoder()
        self.model = LogisticRegression(max_iter=1000)

    def fit(self, df):

        df["text"] = df["competence_clean"] + " " + df["region_clean"]

        X_text = self.vectorizer.fit_transform(df["text"])
        X_region = self.encoder.fit_transform(df[["region_clean"]])

        X = hstack([X_text, X_region])
        y = df["poste"]

        self.model.fit(X, y)
        return self  

    def predict(self, skills, region):
        """
        Prédit un poste à partir des skills + région
        """
        text = clean_text(skills + " " + region)

        X_text = self.vectorizer.transform([text])
        X_region = self.encoder.transform([[clean_text(region)]])

        X = hstack([X_text, X_region])

        return self.model.predict(X)[0]



In [19]:
model1 = TfidfJobModel()
model1.fit(data_prepa)

In [20]:
def entrainement1(donnee):
    model_instance = model1.fit(donnee)
    
    return {
        "model": model_instance.model, 
        "vectorizer": model_instance.vectorizer, 
        "encoder": model_instance.encoder
    }

# entrainement1(data_prepa)


def test(comp, region):
    return model1.predict(comp, region)


In [21]:
test("Python HTML CSS", "dakar")

/home/ben/Bureau/Pre-production/Offres_emplois_senegal/.benv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


'PYTHON Developer (M/F)'

In [14]:
print("Prediction (TF-IDF):")
model1.predict("Python SQL", "Dakar")

Prediction (TF-IDF):
/home/ben/Bureau/Pre-production/Offres_emplois_senegal/.benv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(



'PYTHON Developer (M/F)'

In [ ]:


# =========================================================
# 🤖 3. MODEL 1 — TF-IDF + LOGISTIC REGRESSION
# =========================================================



# =========================================================
# 🧠 4. MODEL 2 — EMBEDDINGS + SIMILARITY (RECOMMANDÉ)
# =========================================================

class EmbeddingJobRecommender:
    
    def __init__(self):
        self.model_emb = SentenceTransformer("all-MiniLM-L6-v2")
        self.df = None

    def fit(self, df):
        """
        Génère les embeddings des jobs
        """
        df["job_profile"] = (
            df["poste"] + " " + df["competence_clean"] + " " + df["region_clean"] + df["experience"]
        )

        df["embedding"] = df["job_profile"].apply(
            lambda x: self.model_emb.encode(x)
        )

        self.df = df

    # -----------------------------
    # SCORING FUNCTIONS
    # -----------------------------

    def skill_score(self, user_emb, job_emb):
        return cosine_similarity([user_emb], [job_emb])[0][0]

    def experience_score(self, user_exp, job_exp):
        diff = abs(user_exp - job_exp)
        return max(0, 1 - diff / 10)

    def region_score(self, user_region, job_region):
        return 1 if user_region == job_region else 0.3

    def hybrid_score(self, skill, exp, region):
        """
        Pondération globale
        """
        return 0.6 * skill + 0.25 * exp + 0.15 * region

    # -----------------------------
    # SKILLS GAP
    # -----------------------------

    def missing_skills(self, user_skills, job_skills):
        user_set = set(user_skills.split())
        job_set = set(job_skills.split())
        return list(job_set - user_set)

    # -----------------------------
    # RECOMMENDATION
    # -----------------------------

    def recommend(self, skills, region, experience, top_k=5):
        """
        Retourne les meilleurs jobs + explication
        """

        user_text = clean_text(skills)
        user_region = clean_text(region)

        user_emb = self.model_emb.encode(user_text)

        results = []

        for _, row in self.df.iterrows():

            s_score = self.skill_score(user_emb, row["embedding"])
            e_score = self.experience_score(experience, row["experience"])
            r_score = self.region_score(user_region, row["region_clean"])

            final_score = self.hybrid_score(s_score, e_score, r_score)

            results.append({
                "poste": row["poste"],
                "region": row["region_clean"],
                "score": final_score,
                "missing_skills": self.missing_skills(user_text, row["competence_clean"])
            })

        results = sorted(results, key=lambda x: x["score"], reverse=True)

        return results[:top_k]

# =========================================================
# 🚀 5. PIPELINE PRINCIPAL
# =========================================================

def run_pipeline(df):
    """
    Pipeline complet :
    - préparation données
    - entraînement modèles
    """

    data = prepare_data(df)

    # modèle 1
    tfidf_model = TfidfJobModel()
    tfidf_model.fit(data)

    # modèle 2
    emb_model = EmbeddingJobRecommender()
    emb_model.fit(data)

    return tfidf_model, emb_model

# =========================================================
# 🧪 6. EXEMPLE D'UTILISATION
# =========================================================

if __name__ == "__main__":
    
    df = pd.read_csv("my_pipeline__run_pipeline__output_1.csv")

    tfidf_model, emb_model = run_pipeline(df)

    # test modèle 1
    print("Prediction (TF-IDF):")
    print(tfidf_model.predict("python sql machine learning", "Dakar"))

    # test modèle 2
    print("\nRecommandations (Embedding):")
    recs = emb_model.recommend(
        skills="python sql machine learning",
        region="Dakar",
        experience=2
    )

    for r in recs:
        print(r)

FileNotFoundError: [Errno 2] No such file or directory: 'my_pipeline__run_pipeline__output_1.csv'

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════╗
║            JOB RECOMMENDATION PIPELINE — ZenML                          ║
║                                                                          ║
║  Steps :                                                                 ║
║    step_clean        → Nettoyage texte brut                              ║
║    step_parse_exp    → Catégorie d'expérience → intervalle [min, max]    ║
║    step_features     → Agrégation & profil job                           ║
║    step_train_tfidf  → Modèle 1 : TF-IDF + LogisticRegression           ║
║    step_train_emb    → Modèle 2 : Embeddings + Scoring Hybride           ║
║    job_pipeline      → Orchestration ZenML                               ║
╚══════════════════════════════════════════════════════════════════════════╝

Catégories d'expérience reconnues (vos données réelles) :
    'Etudiant'
    'jeune diplômé'
    'jeune diplômé et plus'
    'Débutant < 2 ans'
    'Débutant < 2 ans et plus'
    'Expérience entre 2 ans et 5 ans'
    'Expérience entre 2 ans et 5 ans et plus'
    'Expérience entre 5 ans et 10 ans'
    'Expérience > 10 ans'

Installation :
    pip install scikit-learn sentence-transformers pandas numpy zenml
"""

# ──────────────────────────────────────────────────────────────────────────
# IMPORTS
# ──────────────────────────────────────────────────────────────────────────

import re
import logging
from dataclasses import dataclass
from enum import Enum
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder
from sentence_transformers import SentenceTransformer
from zenml import pipeline, step

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)


# ══════════════════════════════════════════════════════════════════════════
# MODÈLE DE DONNÉES — ExperienceLevel
# ══════════════════════════════════════════════════════════════════════════

@dataclass(frozen=True)
class ExperienceRange:
    """
    Représente un niveau d'expérience par un intervalle [min_years, max_years].

    Attributs :
        label     : libellé canonique affiché
        min_years : borne inférieure (incluse)
        max_years : borne supérieure (float('inf') si pas de plafond)
        midpoint  : valeur centrale utilisée dans le scoring
    """
    label    : str
    min_years: float
    max_years: float

    @property
    def midpoint(self) -> float:
        """Point médian de l'intervalle (valeur de référence pour le scoring)."""
        if self.max_years == float("inf"):
            return self.min_years + 3.0   # convention : +3 ans au-delà du seuil
        return (self.min_years + self.max_years) / 2.0


class ExperienceLevel(Enum):
    """
    Énumération exhaustive des niveaux d'expérience du dataset.

    Chaque membre encapsule un ExperienceRange avec min/max en années.
    L'ordre reflète la progression de carrière (utilisé pour la comparaison).

    Niveaux reconnus dans vos données :
        Etudiant                              → [0, 0]
        jeune diplômé                         → [0, 1]
        jeune diplômé et plus                 → [0, inf)
        Débutant < 2 ans                      → [0, 2)
        Débutant < 2 ans et plus              → [0, inf)
        Expérience entre 2 ans et 5 ans       → [2, 5]
        Expérience entre 2 ans et 5 ans et plus → [2, inf)
        Expérience entre 5 ans et 10 ans      → [5, 10]
        Expérience > 10 ans                   → [10, inf)
    """
    ETUDIANT          = ExperienceRange("Etudiant",                              0.0,  0.0 )
    JEUNE_DIPLOME     = ExperienceRange("jeune diplômé",                         0.0,  1.0 )
    JEUNE_DIPLOME_PLUS= ExperienceRange("jeune diplômé et plus",                 0.0,  float("inf"))
    DEBUTANT          = ExperienceRange("Débutant < 2 ans",                      0.0,  2.0 )
    DEBUTANT_PLUS     = ExperienceRange("Débutant < 2 ans et plus",              0.0,  float("inf"))
    EXP_2_5           = ExperienceRange("Expérience entre 2 ans et 5 ans",       2.0,  5.0 )
    EXP_2_5_PLUS      = ExperienceRange("Expérience entre 2 ans et 5 ans et plus", 2.0, float("inf"))
    EXP_5_10          = ExperienceRange("Expérience entre 5 ans et 10 ans",      5.0,  10.0)
    EXP_10_PLUS       = ExperienceRange("Expérience > 10 ans",                   10.0, float("inf"))

    @property
    def midpoint(self) -> float:
        return self.value.midpoint

    @property
    def min_years(self) -> float:
        return self.value.min_years

    @property
    def max_years(self) -> float:
        return self.value.max_years


# ══════════════════════════════════════════════════════════════════════════
# UTILITAIRES — Fonctions pures (non-steps)
# ══════════════════════════════════════════════════════════════════════════

def clean_text(text: str) -> str:
    """
    Normalise un texte brut :
      - passage en minuscule
      - suppression des caractères spéciaux (garde alphanumérique + espaces)
      - collapsing des espaces multiples
    """
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Table de correspondance : pattern regex → ExperienceLevel
# Ordre IMPORTANT : du plus spécifique au plus général pour éviter les faux positifs.
_EXPERIENCE_PATTERNS: List[Tuple[re.Pattern, ExperienceLevel]] = [
    # ── Fourchettes numériques explicites ────────────────────────────────
    (re.compile(r"exp.*5.*10|5\s*[àa]\s*10",         re.I), ExperienceLevel.EXP_5_10      ),
    (re.compile(r"exp.*2.*5.*plus|2.*5.*plus",        re.I), ExperienceLevel.EXP_2_5_PLUS  ),
    (re.compile(r"exp.*2.*5|2\s*[àa]\s*5",           re.I), ExperienceLevel.EXP_2_5       ),
    (re.compile(r"exp.*>\s*10|plus de 10|10.*plus",  re.I), ExperienceLevel.EXP_10_PLUS   ),
    # ── Débutant ─────────────────────────────────────────────────────────
    (re.compile(r"d[ée]butant.*<\s*2.*plus",         re.I), ExperienceLevel.DEBUTANT_PLUS ),
    (re.compile(r"d[ée]butant.*<\s*2|<\s*2\s*ans",  re.I), ExperienceLevel.DEBUTANT      ),
    (re.compile(r"d[ée]butant",                      re.I), ExperienceLevel.DEBUTANT      ),
    # ── Jeune diplômé ────────────────────────────────────────────────────
    (re.compile(r"jeune\s+diplom[eé].*plus",         re.I), ExperienceLevel.JEUNE_DIPLOME_PLUS),
    (re.compile(r"jeune\s+diplom[eé]",               re.I), ExperienceLevel.JEUNE_DIPLOME ),
    # ── Étudiant ─────────────────────────────────────────────────────────
    (re.compile(r"[ée]tudiant|student",              re.I), ExperienceLevel.ETUDIANT      ),
    # ── Mots-clés génériques (fallback) ──────────────────────────────────
    (re.compile(r"senior|exp[eé]riment[eé]|confirm[eé]", re.I), ExperienceLevel.EXP_5_10 ),
    (re.compile(r"expert|lead|principal|directeur",  re.I), ExperienceLevel.EXP_10_PLUS  ),
    (re.compile(r"junior|grad",                      re.I), ExperienceLevel.JEUNE_DIPLOME ),
]


def parse_experience(exp_text: str) -> ExperienceLevel:
    """
    Convertit une catégorie d'expérience (texte) en ExperienceLevel.

    Stratégie de matching (ordre décroissant de priorité) :
      1. Correspondance exacte (insensible à la casse + accents)
      2. Regex patterns du plus spécifique au plus général
      3. Extraction d'un nombre d'années brut → intervalle le plus proche
      4. Fallback → ExperienceLevel.DEBUTANT avec warning

    Args:
        exp_text : catégorie brute du dataset
                   (ex: "Expérience entre 2 ans et 5 ans et plus")

    Returns:
        ExperienceLevel correspondant

    Exemples :
        "Etudiant"                               → ETUDIANT       (mid=0.0)
        "jeune diplômé"                          → JEUNE_DIPLOME  (mid=0.5)
        "Débutant < 2 ans"                       → DEBUTANT       (mid=1.0)
        "Expérience entre 2 ans et 5 ans"        → EXP_2_5        (mid=3.5)
        "Expérience entre 2 ans et 5 ans et plus"→ EXP_2_5_PLUS   (mid=5.0)
        "Expérience entre 5 ans et 10 ans"       → EXP_5_10       (mid=7.5)
        "Expérience > 10 ans"                    → EXP_10_PLUS    (mid=13.0)
    """
    if exp_text is None:
        return ExperienceLevel.DEBUTANT

    raw = str(exp_text).strip()

    # ── 1. Correspondance exacte (robuste aux accents) ────────────────────
    normalized = raw.lower()
    for level in ExperienceLevel:
        if level.value.label.lower() == normalized:
            return level

    # ── 2. Matching regex (ordre : spécifique → général) ─────────────────
    for pattern, level in _EXPERIENCE_PATTERNS:
        if pattern.search(raw):
            return level

    # ── 3. Nombre d'années brut → niveau le plus proche par midpoint ─────
    match = re.search(r"(\d+(?:[.,]\d+)?)\s*ans?", raw, re.I)
    if match:
        years = float(match.group(1).replace(",", "."))
        closest = min(ExperienceLevel, key=lambda lvl: abs(lvl.midpoint - years))
        logger.info(f"   ℹ '{raw}' → {years} ans → {closest.name} (par proximité)")
        return closest

    # ── 4. Fallback ───────────────────────────────────────────────────────
    logger.warning(f"   ⚠ Expérience non reconnue : '{raw}' → défaut DEBUTANT")
    return ExperienceLevel.DEBUTANT


def experience_overlap_score(user_level: ExperienceLevel, job_level: ExperienceLevel) -> float:
    """
    Calcule un score de compatibilité entre le niveau utilisateur et le niveau requis.

    Méthode : chevauchement des intervalles normalisé par la taille de l'intervalle job.
    Avantage : pénalise moins les profils "surqualifiés" qu'une simple distance de midpoints.

    Cas particuliers :
      - Intervalle ouvert (max=inf) → plafonné à midpoint + 5 ans pour le calcul
      - Même niveau exact → score 1.0

    Exemples :
        user=EXP_2_5  / job=EXP_2_5       → 1.00  (match parfait)
        user=EXP_2_5  / job=EXP_5_10      → 0.00  (pas de chevauchement)
        user=EXP_5_10 / job=EXP_2_5_PLUS  → 1.00  (EXP_5_10 ⊂ [2, ∞))
        user=DEBUTANT / job=EXP_5_10      → 0.00  (très éloigné)
    """
    _CAP = 20.0   # années de plafond pour les intervalles ouverts (> 10 ans)

    u_min = user_level.min_years
    u_max = min(user_level.max_years, _CAP) if user_level.max_years == float("inf") \
            else user_level.max_years

    j_min = job_level.min_years
    j_max = min(job_level.max_years, _CAP) if job_level.max_years == float("inf") \
            else job_level.max_years

    # Cas dégénéré : intervalle ponctuel (Etudiant → [0,0])
    if j_max == j_min:
        return 1.0 if (u_min <= j_min <= u_max) else max(0.0, 1.0 - abs(u_min - j_min) / 5.0)

    # Chevauchement des intervalles
    overlap = max(0.0, min(u_max, j_max) - max(u_min, j_min))
    job_span = j_max - j_min

    return round(overlap / job_span, 4) if job_span > 0 else 0.0


# ══════════════════════════════════════════════════════════════════════════
# STEP 1 — NETTOYAGE
# ══════════════════════════════════════════════════════════════════════════

@step
def step_clean(df: pd.DataFrame) -> pd.DataFrame:
    """
    ZenML Step — Nettoyage des colonnes textuelles.

    Input  : DataFrame brut [poste, competence, region, experience]
    Output : DataFrame avec colonnes _clean suffixées
    """
    logger.info("🧹 Step 1 — Nettoyage des données...")

    data = df[["poste", "competence", "region", "experience"]].copy()
    data["competence_clean"] = data["competence"].apply(clean_text)
    data["region_clean"]     = data["region"].apply(clean_text)

    logger.info(f"   ✔ {len(data)} lignes nettoyées.")
    return data


# ══════════════════════════════════════════════════════════════════════════
# STEP 2 — PARSING DE L'EXPÉRIENCE
# ══════════════════════════════════════════════════════════════════════════

@step
def step_parse_exp(data: pd.DataFrame) -> pd.DataFrame:
    """
    ZenML Step — Conversion des catégories d'expérience en ExperienceLevel.

    Ajoute trois colonnes :
      experience_level   : nom de l'enum (ex: "EXP_2_5")
      experience_midpoint: point médian en années (ex: 3.5)
      experience_min     : borne inférieure (ex: 2.0)
      experience_max     : borne supérieure (ex: 5.0, ou 20.0 si ouvert)

    Input  : DataFrame nettoyé (colonne 'experience' brute)
    Output : DataFrame enrichi des colonnes ci-dessus
    """
    logger.info("📐 Step 2 — Parsing des niveaux d'expérience...")

    data = data.copy()

    levels = data["experience"].apply(parse_experience)

    data["experience_level"]    = levels.apply(lambda l: l.name)
    data["experience_midpoint"] = levels.apply(lambda l: l.midpoint)
    data["experience_min"]      = levels.apply(lambda l: l.min_years)
    data["experience_max"]      = levels.apply(
        lambda l: l.max_years if l.max_years != float("inf") else 20.0
    )

    # Distribution pour monitoring
    dist = data["experience_level"].value_counts()
    logger.info(f"   ✔ Distribution des niveaux :\n{dist.to_string()}")

    return data


# ══════════════════════════════════════════════════════════════════════════
# STEP 3 — FEATURES : AGRÉGATION PAR POSTE + RÉGION
# ══════════════════════════════════════════════════════════════════════════

@step
def step_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    ZenML Step — Agrégation par (poste, région).

    Produit :
      - competence_clean   : toutes les compétences du groupe concaténées
      - experience_raw     : valeurs brutes originales (traçabilité)
      - experience_min_avg : moyenne des bornes inférieures
      - experience_max_avg : moyenne des bornes supérieures
      - experience_mid_avg : moyenne des midpoints (utilisé dans le scoring)
      - experience_levels  : liste des niveaux uniques du groupe
      - job_profile        : texte agrégé pour les embeddings

    Input  : DataFrame step_parse_exp
    Output : 1 ligne = 1 profil job unique (poste × région)
    """
    logger.info("⚙️  Step 3 — Construction des features...")

    GROUP_KEYS = ["poste", "region_clean"]

    # ── Agrégation compétences ────────────────────────────────────────────
    competences_agg = (
        data.groupby(GROUP_KEYS)["competence_clean"]
        .apply(lambda x: " ".join(x))
        .dropna()
        .reset_index()
    )

    # ── Agrégation expérience brute (traçabilité) ─────────────────────────
    exp_raw_agg = (
        data.groupby(GROUP_KEYS)["experience"]
        .apply(lambda x: " | ".join(x.astype(str).unique()))
        .reset_index()
        .rename(columns={"experience": "experience_raw"})
    )

    # ── Niveaux d'expérience uniques du groupe ────────────────────────────
    exp_levels_agg = (
        data.groupby(GROUP_KEYS)["experience_level"]
        .apply(lambda x: ", ".join(sorted(x.unique())))
        .reset_index()
        .rename(columns={"experience_level": "experience_levels"})
    )

    # ── Statistiques numériques sur l'intervalle ──────────────────────────
    exp_stats_agg = (
        data.groupby(GROUP_KEYS)[["experience_min", "experience_max", "experience_midpoint"]]
        .mean()
        .reset_index()
        .rename(columns={
            "experience_min":      "experience_min_avg",
            "experience_max":      "experience_max_avg",
            "experience_midpoint": "experience_mid_avg",
        })
    )

    # ── Fusion ────────────────────────────────────────────────────────────
    grouped = (
        competences_agg
        .merge(exp_raw_agg,    on=GROUP_KEYS)
        .merge(exp_levels_agg, on=GROUP_KEYS)
        .merge(exp_stats_agg,  on=GROUP_KEYS)
    )

    # ── Profil textuel complet (pour les embeddings) ──────────────────────
    grouped["job_profile"] = (
        grouped["poste"] + " "
        + grouped["competence_clean"] + " "
        + grouped["region_clean"]
    )

    logger.info(f"   ✔ {len(grouped)} profils de jobs construits.")
    return grouped


# ══════════════════════════════════════════════════════════════════════════
# MODÈLE 1 — TF-IDF + LOGISTIC REGRESSION (BASELINE)
# ══════════════════════════════════════════════════════════════════════════

class TfidfJobModel:
    """
    Classificateur de poste : TF-IDF + OneHotEncoder (région) + LogisticRegression.

    Usage type  : prédiction rapide d'un intitulé de poste depuis texte libre
    Avantages   : léger, interprétable, pas de GPU
    Limites     : sensible au vocabulaire exact, pas de compréhension sémantique
    """

    def __init__(self, max_features: int = 5000):
        self.vectorizer = TfidfVectorizer(max_features=max_features)
        self.encoder    = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
        self.model      = LogisticRegression(max_iter=1000, class_weight="balanced")

    def fit(self, df: pd.DataFrame) -> None:
        """
        Entraîne le pipeline TF-IDF → OneHot → LogReg.

        Colonnes requises : competence_clean, region_clean, poste
        """
        df = df.copy()
        df["text"] = df["competence_clean"] + " " + df["region_clean"]

        X_text   = self.vectorizer.fit_transform(df["text"])
        X_region = self.encoder.fit_transform(df[["region_clean"]])
        X        = hstack([X_text, X_region])
        y        = df["poste"]

        self.model.fit(X, y)

    def predict(self, skills: str, region: str) -> str:
        """
        Prédit le poste le plus probable.

        Args:
            skills : compétences brutes (ex: "python sql machine learning")
            region : région brute       (ex: "Dakar")
        Returns:
            Intitulé du poste prédit (str)
        """
        text     = clean_text(skills + " " + region)
        X_text   = self.vectorizer.transform([text])
        X_region = self.encoder.transform([[clean_text(region)]])
        X        = hstack([X_text, X_region])
        return self.model.predict(X)[0]


@step
def step_train_tfidf(df: pd.DataFrame) -> TfidfJobModel:
    """
    ZenML Step — Entraînement du modèle TF-IDF baseline.

    Input  : DataFrame agrégé (step_features)
    Output : TfidfJobModel entraîné
    """
    logger.info("🤖 Step 4a — Entraînement TF-IDF + LogisticRegression...")
    model = TfidfJobModel()
    model.fit(df)
    logger.info("   ✔ Modèle 1 prêt.")
    return model


# ══════════════════════════════════════════════════════════════════════════
# MODÈLE 2 — EMBEDDINGS + SCORING HYBRIDE (PRODUCTION)
# ══════════════════════════════════════════════════════════════════════════

class EmbeddingJobRecommender:
    """
    Moteur de recommandation sémantique avec scoring hybride.

    Architecture :
      - SentenceTransformer → embeddings des profils jobs
      - Scoring hybride pondéré :
          60% similarité compétences     (cosinus entre embeddings)
          25% compatibilité expérience   (chevauchement d'intervalles)
          15% correspondance région      (exacte ou mobilité partielle)
      - Détection des compétences manquantes (skills gap)

    Avantages   : robuste au vocabulaire, compréhension sémantique
    Limites     : plus lent (~80 Mo de modèle), O(n) à l'inférence
    """

    EMB_MODEL_NAME = "all-MiniLM-L6-v2"

    # Pondérations du scoring hybride — doivent sommer à 1.0
    W_SKILLS     = 0.60
    W_EXPERIENCE = 0.25
    W_REGION     = 0.15

    def __init__(self):
        logger.info(f"📦 Chargement embedding model : {self.EMB_MODEL_NAME}")
        self.model_emb = SentenceTransformer(self.EMB_MODEL_NAME)
        self.df: Optional[pd.DataFrame] = None

    # ── FIT ──────────────────────────────────────────────────────────────

    def fit(self, df: pd.DataFrame) -> None:
        """
        Génère et stocke les embeddings de chaque profil de job.

        Colonne requise : job_profile (texte agrégé poste + compétences + région)
        """
        df = df.copy()
        df["embedding"] = df["job_profile"].apply(self.model_emb.encode)
        self.df = df

    # ── SCORING ──────────────────────────────────────────────────────────

    def _skill_score(self, user_emb: np.ndarray, job_emb: np.ndarray) -> float:
        """Similarité cosinus entre l'embedding utilisateur et le profil job."""
        return float(cosine_similarity([user_emb], [job_emb])[0][0])

    def _experience_score(
        self, user_level: ExperienceLevel, job_row: pd.Series
    ) -> float:
        """
        Compatibilité d'expérience par chevauchement d'intervalles.

        Stratégie :
          1. Si le job a un niveau d'expérience unique → overlap direct
          2. Si le job regroupe plusieurs niveaux → score max sur tous les niveaux
             (un profil est retenu si au moins un niveau du job lui correspond)

        Args:
            user_level : ExperienceLevel de l'utilisateur
            job_row    : ligne du DataFrame jobs (contient experience_levels)
        """
        # Reconstitue les ExperienceLevel du job à partir de la colonne texte
        level_names = [s.strip() for s in job_row["experience_levels"].split(",")]
        scores = []
        for name in level_names:
            try:
                job_level = ExperienceLevel[name]
                scores.append(experience_overlap_score(user_level, job_level))
            except KeyError:
                continue
        return max(scores) if scores else 0.0

    def _region_score(self, user_region: str, job_region: str) -> float:
        """
        Score géographique :
          1.0 → même région
          0.3 → région différente (mobilité partielle modélisée)
        """
        return 1.0 if user_region == job_region else 0.3

    def _hybrid_score(self, s: float, e: float, r: float) -> float:
        """Agrégation pondérée des trois composantes du score."""
        return self.W_SKILLS * s + self.W_EXPERIENCE * e + self.W_REGION * r

    # ── SKILLS GAP ───────────────────────────────────────────────────────

    def missing_skills(self, user_skills: str, job_skills: str) -> List[str]:
        """
        Retourne les mots-clés du job absents du profil utilisateur.

        Note : comparaison token-level après clean_text (split par espace).
        Pour une approche plus sémantique, remplacez par un matcher TF-IDF ou embedding.
        """
        user_set = set(user_skills.split())
        job_set  = set(job_skills.split())
        return sorted(job_set - user_set)

    # ── RECOMMEND ─────────────────────────────────────────────────────────

    def recommend(
        self,
        skills    : str,
        region    : str,
        experience: str,     # catégorie exacte ou texte libre
        top_k     : int = 5,
    ) -> List[dict]:
        """
        Retourne les top_k meilleures recommandations de postes.

        Args:
            skills     : compétences brutes
                         (ex: "python sql machine learning")
            region     : région brute
                         (ex: "Dakar")
            experience : catégorie du dataset ou texte libre
                         (ex: "Expérience entre 2 ans et 5 ans", "senior", "3 ans")
            top_k      : nombre de résultats souhaités

        Returns:
            Liste de dicts triés par score décroissant :
            {
                "poste"            : str,
                "region"           : str,
                "experience_levels": str,
                "score"            : float,
                "score_detail"     : {
                    "skills"      : float,
                    "experience"  : float,
                    "region"      : float,
                },
                "missing_skills"   : list[str],
                "user_exp_level"   : str,
            }
        """
        if self.df is None:
            raise RuntimeError("Modèle non entraîné — appelez .fit() d'abord.")

        user_text    = clean_text(skills)
        user_region  = clean_text(region)
        user_level   = parse_experience(experience)
        user_emb     = self.model_emb.encode(user_text)

        results = []

        for _, row in self.df.iterrows():
            s = self._skill_score(user_emb, row["embedding"])
            e = self._experience_score(user_level, row)
            r = self._region_score(user_region, row["region_clean"])

            results.append({
                "poste"            : row["poste"],
                "region"           : row["region_clean"],
                "experience_levels": row["experience_levels"],
                "score"            : round(self._hybrid_score(s, e, r), 4),
                "score_detail"     : {
                    "skills"    : round(s, 4),
                    "experience": round(e, 4),
                    "region"    : round(r, 4),
                },
                "missing_skills"   : self.missing_skills(user_text, row["competence_clean"]),
                "user_exp_level"   : user_level.value.label,
            })

        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:top_k]


@step
def step_train_emb(df: pd.DataFrame) -> EmbeddingJobRecommender:
    """
    ZenML Step — Génération des embeddings et construction du moteur.

    Input  : DataFrame agrégé (step_features)
    Output : EmbeddingJobRecommender entraîné
    """
    logger.info("🤖 Step 4b — Génération des embeddings...")
    model = EmbeddingJobRecommender()
    model.fit(df)
    logger.info(f"   ✔ {len(df)} embeddings générés.")
    return model


# ══════════════════════════════════════════════════════════════════════════
# PIPELINE ZENML
# ══════════════════════════════════════════════════════════════════════════

@pipeline
def job_pipeline(df: pd.DataFrame):
    """
    Pipeline ZenML complet :

        step_clean
             ↓
        step_parse_exp
             ↓
        step_features
             ↓                ↓
        step_train_tfidf  step_train_emb
    """
    data_clean    = step_clean(df)
    data_exp      = step_parse_exp(data_clean)
    data_features = step_features(data_exp)
    tfidf_model   = step_train_tfidf(data_features)
    emb_model     = step_train_emb(data_features)
    return tfidf_model, emb_model


# ══════════════════════════════════════════════════════════════════════════
# POINT D'ENTRÉE
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # ── Chargement ────────────────────────────────────────────────────────
    df = pd.read_csv("my_pipeline__run_pipeline__output_1.csv")

    # ── Exécution pipeline ────────────────────────────────────────────────
    tfidf_model, emb_model = job_pipeline(df)

    # ── Test exhaustif parse_experience ───────────────────────────────────
    print("\n" + "═" * 70)
    print("  TEST — parse_experience() sur toutes les catégories du dataset")
    print("═" * 70)

    ALL_CATEGORIES = [
        "Etudiant",
        "jeune diplômé",
        "jeune diplômé et plus",
        "Débutant < 2 ans",
        "Débutant < 2 ans et plus",
        "Expérience entre 2 ans et 5 ans",
        "Expérience entre 2 ans et 5 ans et plus",
        "Expérience entre 5 ans et 10 ans",
        "Expérience > 10 ans",
    ]
    for cat in ALL_CATEGORIES:
        lvl = parse_experience(cat)
        max_str = "∞" if lvl.max_years == float("inf") else str(lvl.max_years)
        print(
            f"  {cat:<45} → {lvl.name:<20} "
            f"[{lvl.min_years}, {max_str}]  mid={lvl.midpoint}"
        )

    # ── Test score de compatibilité d'expérience ─────────────────────────
    print("\n" + "═" * 70)
    print("  TEST — experience_overlap_score()")
    print("═" * 70)

    pairs = [
        ("Expérience entre 2 ans et 5 ans", "Expérience entre 2 ans et 5 ans"),
        ("Expérience entre 2 ans et 5 ans", "Expérience entre 5 ans et 10 ans"),
        ("Expérience entre 5 ans et 10 ans", "Expérience entre 2 ans et 5 ans et plus"),
        ("Etudiant",                          "Expérience entre 5 ans et 10 ans"),
        ("Débutant < 2 ans",                  "Débutant < 2 ans et plus"),
    ]
    for u, j in pairs:
        u_lvl = parse_experience(u)
        j_lvl = parse_experience(j)
        score = experience_overlap_score(u_lvl, j_lvl)
        print(f"  user={u_lvl.name:<20}  job={j_lvl.name:<20}  → score={score:.4f}")

    # ── Test Modèle 1 ─────────────────────────────────────────────────────
    print("\n" + "═" * 70)
    print("  MODÈLE 1 — TF-IDF + LogisticRegression")
    print("═" * 70)
    pred = tfidf_model.predict("python sql machine learning", "Dakar")
    print(f"  Poste prédit : {pred}")

    # ── Test Modèle 2 ─────────────────────────────────────────────────────
    print("\n" + "═" * 70)
    print("  MODÈLE 2 — Embeddings + Scoring Hybride")
    print("═" * 70)
    recs = emb_model.recommend(
        skills="python sql machine learning",
        region="Dakar",
        experience="Expérience entre 2 ans et 5 ans",   # ← catégorie exacte
        top_k=5,
    )
    for i, rec in enumerate(recs, 1):
        print(f"\n  #{i}  {rec['poste']} ({rec['region']})")
        print(f"       Niveaux requis  : {rec['experience_levels']}")
        print(f"       Profil user     : {rec['user_exp_level']}")
        print(f"       Score global    : {rec['score']:.4f}  "
              f"[skills={rec['score_detail']['skills']:.2f} | "
              f"exp={rec['score_detail']['experience']:.2f} | "
              f"région={rec['score_detail']['region']:.2f}]")
        missing = ", ".join(rec["missing_skills"][:5]) or "aucune ✔"
        print(f"       Compétences manquantes : {missing}")

    print("\n" + "═" * 70 + "\n")